In [0]:
from pyspark.sql.functions import coalesce, col
spark.sql("USE CATALOG biking_product_sales_lakehouse")
spark.sql("USE SCHEMA silver_staging")

In [0]:
# Cusomter entity joining
df_customer_crm = spark.read.table("biking_product_sales_lakehouse.silver_staging.customer_crm")
df_customer_erp = spark.read.table("biking_product_sales_lakehouse.silver_staging.customer_erp")
df_customer_location = spark.read.table("biking_product_sales_lakehouse.silver_staging.customer_location")
df_customers_crm = df_customer_crm.\
    join(df_customer_location, on="customer_key", how="full_outer")

In [0]:
# Product entity
df_product_crm = spark.read.table("biking_product_sales_lakehouse.silver_staging.product_crm")
df_product_category = spark.read.table("biking_product_sales_lakehouse.silver_staging.product_cat_subcat")

In [0]:
# Sales
df_sales = spark.read.table("biking_product_sales_lakehouse.silver_staging.sales_crm")

In [0]:
entities = {
    "customer_crm": df_customers_crm,
    "customer_erp": df_customer_erp,
    "products": df_product_crm,
    "product_category": df_product_category,
    "sales": df_sales
}

In [0]:
for key, df in entities.items():
    df.write.format("delta").\
        mode("overwrite").\
        saveAsTable(f"biking_product_sales_lakehouse.silver_curated.{key}")